# 評估 LLM 品質 — 自動化品質檢測

**目標**:準備評估資料集、執行自動化評估、自訂評估指標

**前置條件**:
- 已完成 [03_full_workflow.ipynb](03_full_workflow.ipynb)
- 理解 Workflow 的執行流程


In [ ]:
from llm_framework.config import load_config
from llm_framework.llm_client import LLMClient
from llm_framework.evaluation.evaluator import Evaluator
import pandas as pd

load_config("dev")


## 準備評估資料集

建立包含問題和預期答案的資料集。


In [ ]:
eval_data = pd.DataFrame([
    {"question": "什麼是機器學習?", "expected": "機器學習是人工智慧的一個分支,讓電腦從資料中學習"},
    {"question": "Python 是什麼?", "expected": "Python 是一種高階程式語言"},
    {"question": "什麼是深度學習?", "expected": "深度學習是機器學習的子領域,使用神經網路"},
    {"question": "什麼是 API?", "expected": "API 是應用程式介面,用於不同軟體間的溝通"},
    {"question": "什麼是雲端運算?", "expected": "雲端運算是透過網路提供運算資源和服務"}
])

print(f"資料集大小: {len(eval_data)} 筆")
eval_data.head()


## 產生 LLM 回答


In [ ]:
client = LLMClient()
responses = []

for question in eval_data["question"]:
    response = client.chat(
        [{"role": "user", "content": f"簡短回答:{question}"}],
        temperature=0.0,
        max_tokens=100
    )
    responses.append(response.content)

eval_data["actual"] = responses
print("已產生所有回答")
eval_data.head()


## 執行評估


In [ ]:
evaluator = Evaluator()
results = evaluator.evaluate(
    eval_data,
    metrics=["exact_match", "semantic_similarity"],
    expected_col="expected",
    actual_col="actual"
)

print("評估完成")


## 查看結果


In [ ]:
print("整體指標:")
for metric, value in results["metrics"].items():
    print(f"  {metric}: {value:.3f}")

print("\n個別結果:")
for i, row in enumerate(results["per_row"][:3]):
    print(f"\n問題 {i+1}: {eval_data.iloc[i]['question']}")
    print(f"  預期: {eval_data.iloc[i]['expected'][:50]}...")
    print(f"  實際: {eval_data.iloc[i]['actual'][:50]}...")
    print(f"  分數: {row}")


## 自訂評估指標

定義自己的評估邏輯。


In [ ]:
def length_check(expected: str, actual: str) -> float:
    """檢查回答長度是否適中(20-200字元)"""
    length = len(actual)
    if 20 <= length <= 200:
        return 1.0
    return 0.0

# 使用自訂指標
custom_results = evaluator.evaluate(
    eval_data,
    metrics=["exact_match", length_check],
    expected_col="expected",
    actual_col="actual"
)

print("自訂評估結果:")
for metric, value in custom_results["metrics"].items():
    print(f"  {metric}: {value:.3f}")


## 總結

你已經學會了:
- 準備評估資料集
- 使用 Evaluator 執行自動化評估
- 自訂評估指標和邏輯
- 分析評估結果

**下一步**:[05_multi_env.ipynb](05_multi_env.ipynb) — 多環境部署
